# Experiment Management and Comparison

## Notebook Contract
- **Objective:** demonstrate the v0.2 experiment-management workflow: explicit run IDs, persisted training configs, dataset hashes, repeated train/eval runs, and side-by-side comparison tables.
- **Inputs:** synthetic support-ticket data generated locally (no Hub downloads).
- **Outputs:** one JSON run record per configuration under `reports/sample_run/runs/`, a comparison table, a per-class report, and a confusion-matrix heatmap for the best run.
- **Expected runtime:** under 1 minute on CPU with default settings.
- **Scope boundary:** production logic stays in `src/hf_finetuning_lab/experiments`; this notebook orchestrates and visualizes.

## 1) Setup and Reproducibility

In [1]:
from __future__ import annotations

import os
import platform
import random
import sys
from dataclasses import dataclass
from datetime import UTC, datetime
from pathlib import Path

ROOT = Path('..').resolve()
SRC_PATH = ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from hf_finetuning_lab.data.io import build_label_mapping, validate_text_classification_frame
from hf_finetuning_lab.evaluation.metrics import confusion_matrix_frame, per_class_report
from hf_finetuning_lab.experiments import (
    RunRecord,
    hash_dataframe,
    load_runs,
    make_run_id,
    runs_to_frame,
    save_run,
)
from hf_finetuning_lab.model_cards.model_card import write_model_card
from hf_finetuning_lab.sample_data import write_sample_data

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print(f'Python: {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
print(f"Timestamp (UTC): {datetime.now(UTC).isoformat(timespec='seconds')}")
print(f'Seed: {SEED}')


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\diogo\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\diogo\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\diogo\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 758, in start
    self.io_lo

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\diogo\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\diogo\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\diogo\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 758, in start
    self.io_lo

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



Python: 3.12.11
Platform: Windows-11-10.0.26200-SP0
Timestamp (UTC): 2026-05-17T20:09:52+00:00
Seed: 42


## 2) Parameters

The sweep below stays small so the notebook runs end-to-end on a CPU. Increase `ROWS` or extend `CONFIGURATIONS` to push harder.

In [2]:
DATA_PATH = ROOT / 'data' / 'raw' / 'support_tickets.csv'
REPORTS_DIR = ROOT / 'reports' / 'sample_run'
RUNS_DIR = REPORTS_DIR / 'runs'
COMPARISON_PATH = REPORTS_DIR / 'experiment_comparison.csv'
BEST_MODEL_CARD_PATH = REPORTS_DIR / 'best_run_model_card.md'

TEXT_COL = 'text'
LABEL_COL = 'label'
ROWS = 800
TEST_SIZE = 0.2

CONFIGURATIONS: list[dict[str, object]] = [
    {'name': 'tfidf-unigram', 'ngram_range': (1, 1), 'max_features': 8000, 'C': 1.0},
    {'name': 'tfidf-bigram', 'ngram_range': (1, 2), 'max_features': 12000, 'C': 1.0},
    {'name': 'tfidf-bigram-strong', 'ngram_range': (1, 2), 'max_features': 12000, 'C': 4.0},
    {'name': 'tfidf-trigram', 'ngram_range': (1, 3), 'max_features': 20000, 'C': 1.0},
]

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Root: {ROOT}')
print(f'Runs dir: {RUNS_DIR}')
print(f'Configurations: {len(CONFIGURATIONS)}')

Root: C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab
Runs dir: C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run\runs
Configurations: 4


## 3) Data Loading, Validation, and Hashing

The dataset hash is recorded with every run so we can tell at a glance whether two runs trained on identical data.

In [3]:
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
write_sample_data(output=DATA_PATH, rows=ROWS, seed=SEED)

df = pd.read_csv(DATA_PATH)
validate_text_classification_frame(df, text_col=TEXT_COL, label_col=LABEL_COL)

label2id, id2label = build_label_mapping(df[LABEL_COL])
DATASET_HASH = hash_dataframe(df, columns=[TEXT_COL, LABEL_COL])

print(f'Rows: {len(df)}')
print(f'Labels: {list(label2id.keys())}')
print(f'Dataset hash: {DATASET_HASH[:16]}…')

Rows: 800
Labels: ['account', 'billing', 'cancellation', 'delivery', 'security', 'technical']
Dataset hash: fafc0c83a222c6c5…


In [4]:
train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df[LABEL_COL],
)
print(f'Train rows: {len(train_df)} | Test rows: {len(test_df)}')

Train rows: 640 | Test rows: 160


## 4) Single Run: Train, Evaluate, Persist

`train_and_record` encapsulates one training/evaluation cycle. Each call produces a deterministic `RunRecord` (run ID, config, dataset hash, metrics, per-class breakdown) and writes it to `reports/sample_run/runs/`.

In [5]:
@dataclass(slots=True)
class RunOutput:
    record: RunRecord
    predictions: np.ndarray


def train_and_record(config: dict[str, object]) -> RunOutput:
    """Train one TF-IDF + LogReg pipeline and persist its run record."""
    pipeline = Pipeline(
        steps=[
            (
                'tfidf',
                TfidfVectorizer(
                    ngram_range=tuple(config['ngram_range']),
                    max_features=int(config['max_features']),
                ),
            ),
            ('clf', LogisticRegression(max_iter=1000, C=float(config['C']), random_state=SEED)),
        ]
    )
    pipeline.fit(train_df[TEXT_COL], train_df[LABEL_COL])
    pred = pipeline.predict(test_df[TEXT_COL])

    macro_f1 = float(f1_score(test_df[LABEL_COL], pred, average='macro'))
    weighted_f1 = float(f1_score(test_df[LABEL_COL], pred, average='weighted'))
    per_class = per_class_report(test_df[LABEL_COL].to_numpy(), pred).to_dict(orient='index')

    record = RunRecord(
        run_id=make_run_id(prefix=str(config['name'])),
        task='text-classification',
        model_name='tfidf-logreg',
        dataset_hash=DATASET_HASH,
        params={
            'ngram_range': list(config['ngram_range']),
            'max_features': int(config['max_features']),
            'C': float(config['C']),
            'test_size': TEST_SIZE,
            'seed': SEED,
        },
        metrics={'macro_f1': macro_f1, 'weighted_f1': weighted_f1},
        per_class={str(k): {str(mk): float(mv) for mk, mv in v.items()} for k, v in per_class.items()},
        notes=str(config['name']),
    )
    save_run(record, RUNS_DIR)
    return RunOutput(record=record, predictions=pred)


demo = train_and_record(CONFIGURATIONS[0])
print(f'Run ID:      {demo.record.run_id}')
print(f'Macro F1:    {demo.record.metrics["macro_f1"]:.4f}')
print(f'Weighted F1: {demo.record.metrics["weighted_f1"]:.4f}')

Run ID:      tfidf-unigram-20260517T200952Z-ad051c
Macro F1:    1.0000
Weighted F1: 1.0000


## 5) Repeated Runs

Run every configuration in `CONFIGURATIONS`. Each cycle writes a new JSON record under `RUNS_DIR`; previously persisted runs from earlier notebook executions are kept so the comparison table grows over time.

In [6]:
outputs: dict[str, RunOutput] = {}
for config in CONFIGURATIONS:
    output = train_and_record(config)
    outputs[output.record.run_id] = output
    print(
        f"{output.record.notes:<22} macro_f1={output.record.metrics['macro_f1']:.4f} "
        f"weighted_f1={output.record.metrics['weighted_f1']:.4f}"
    )

tfidf-unigram          macro_f1=1.0000 weighted_f1=1.0000
tfidf-bigram           macro_f1=1.0000 weighted_f1=1.0000
tfidf-bigram-strong    macro_f1=1.0000 weighted_f1=1.0000
tfidf-trigram          macro_f1=1.0000 weighted_f1=1.0000


## 6) Comparison Table

Loading from disk (rather than from the in-memory `outputs` dict) means this section also reflects runs from earlier notebook executions — useful for tracking improvements over time.

In [7]:
records = load_runs(RUNS_DIR)
comparison = runs_to_frame(records)
comparison_sorted = comparison.sort_values('metric_macro_f1', ascending=False).reset_index(drop=True)
comparison_sorted.to_csv(COMPARISON_PATH, index=False)
print(f'Loaded {len(records)} run(s) from {RUNS_DIR}')
print(f'Comparison written to {COMPARISON_PATH}')
comparison_sorted[
    [
        'run_id',
        'notes',
        'param_ngram_range',
        'param_max_features',
        'param_C',
        'metric_macro_f1',
        'metric_weighted_f1',
    ]
]

Loaded 10 run(s) from C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run\runs
Comparison written to C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run\experiment_comparison.csv


,run_id,notes,param_ngram_range,param_max_features,param_C,metric_macro_f1,metric_weighted_f1
0,tfidf-bigram-20260513T135149Z-82f158,tfidf-bigram,"[1, 2]",12000,1.0,1.0,1.0
1,tfidf-bigram-strong-20260513T135149Z-588df7,tfidf-bigram-strong,"[1, 2]",12000,4.0,1.0,1.0
2,tfidf-unigram-20260513T135149Z-738cd6,tfidf-unigram,"[1, 1]",8000,1.0,1.0,1.0
3,tfidf-unigram-20260513T135149Z-cab132,tfidf-unigram,"[1, 1]",8000,1.0,1.0,1.0
4,tfidf-trigram-20260513T135150Z-7105a7,tfidf-trigram,"[1, 3]",20000,1.0,1.0,1.0
5,tfidf-bigram-20260517T200952Z-bb386f,tfidf-bigram,"[1, 2]",12000,1.0,1.0,1.0
6,tfidf-bigram-strong-20260517T200952Z-32655c,tfidf-bigram-strong,"[1, 2]",12000,4.0,1.0,1.0
7,tfidf-trigram-20260517T200952Z-1f8437,tfidf-trigram,"[1, 3]",20000,1.0,1.0,1.0
8,tfidf-unigram-20260517T200952Z-43dd31,tfidf-unigram,"[1, 1]",8000,1.0,1.0,1.0
9,tfidf-unigram-20260517T200952Z-ad051c,tfidf-unigram,"[1, 1]",8000,1.0,1.0,1.0


## 7) Best-Run Drill-Down

Pick the run with the highest macro F1 and inspect its per-class report and confusion matrix.

In [8]:
best_row = comparison_sorted.iloc[0]
best_run_id = best_row['run_id']
best_output = outputs.get(best_run_id)
if best_output is None:
    print(f'Best run {best_run_id} was not produced in this session; rerun the sweep to inspect it.')
best_run_id, float(best_row['metric_macro_f1'])

Best run tfidf-bigram-20260513T135149Z-82f158 was not produced in this session; rerun the sweep to inspect it.


('tfidf-bigram-20260513T135149Z-82f158', 1.0)

In [9]:
if best_output is not None:
    per_class_df = per_class_report(test_df[LABEL_COL].to_numpy(), best_output.predictions)
    display_df = per_class_df.round(4)
else:
    display_df = pd.DataFrame()
display_df

""


In [10]:
if best_output is not None:
    labels_sorted = sorted(test_df[LABEL_COL].unique().tolist())
    id2label_for_plot = {idx: name for idx, name in enumerate(labels_sorted)}
    label2id_for_plot = {name: idx for idx, name in id2label_for_plot.items()}
    y_true_ids = test_df[LABEL_COL].map(label2id_for_plot).to_numpy()
    y_pred_ids = np.array([label2id_for_plot[p] for p in best_output.predictions])
    cm_df = confusion_matrix_frame(y_true_ids, y_pred_ids, id2label_for_plot)

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm_df.values, cmap='Blues')
    ax.set_xticks(range(len(cm_df.columns)))
    ax.set_xticklabels(cm_df.columns, rotation=45, ha='right')
    ax.set_yticks(range(len(cm_df.index)))
    ax.set_yticklabels(cm_df.index)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'Confusion matrix — {best_run_id}')
    for i in range(cm_df.shape[0]):
        for j in range(cm_df.shape[1]):
            ax.text(j, i, int(cm_df.values[i, j]), ha='center', va='center', color='black')
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    plt.show()
else:
    print('No in-memory best run output to plot.')

No in-memory best run output to plot.


## 8) Model Card for the Best Run

In [11]:
best_metrics = {
    'macro_f1': float(best_row['metric_macro_f1']),
    'weighted_f1': float(best_row['metric_weighted_f1']),
}
limitations = [
    'Dataset labels are synthetic and do not represent a real operational taxonomy.',
    'Metrics are from a synthetic dataset and are not a proxy for production performance.',
    f'Best run selected by macro F1 from {len(records)} candidates on a single test split; no cross-validation.',
]

card_path = write_model_card(
    output_path=BEST_MODEL_CARD_PATH,
    model_name=f'tfidf-logreg ({best_row["notes"]})',
    task='text-classification',
    label_names=sorted(test_df[LABEL_COL].unique().tolist()),
    metrics=best_metrics,
    limitations=limitations,
)
print(f'Wrote model card to: {card_path}')
print(card_path.read_text(encoding='utf-8')[:900])

Wrote model card to: C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run\best_run_model_card.md
# Model Card: tfidf-logreg (tfidf-bigram)

## Overview

- **Base model:** `tfidf-logreg (tfidf-bigram)`
- **Task:** text-classification
- **Created:** 2026-05-17

## Labels

- account
- billing
- cancellation
- delivery
- security
- technical

## Evaluation Metrics

- **macro_f1**: 1.0000
- **weighted_f1**: 1.0000

## Intended Use

This model is intended for text-classification experiments and workflow demonstrations. Real deployment requires validation on the target domain.

## Limitations

- Dataset labels are synthetic and do not represent a real operational taxonomy.
- Metrics are from a synthetic dataset and are not a proxy for production performance.
- Best run selected by macro F1 from 10 candidates on a single test split; no cross-validation.

## Ethical Considerations

Review privacy, label quality, subgroup performance, and operational failure modes bef

## 9) Checklist
- Each run was written to `reports/sample_run/runs/<run_id>.json` with its config, dataset hash, metrics, and per-class breakdown.
- The comparison table at `reports/sample_run/experiment_comparison.csv` is the source of truth for cross-run analysis.
- Re-running this notebook accumulates additional runs in `RUNS_DIR` rather than replacing them.
- For production: replace the synthetic data path with a real source, add cross-validation, and include subgroup metrics before promoting any single run.